Previously, semantic work was done to proccess profiles intrests. Here, personas are created to cluster the problem over controlled concepts to be used for Neo4j KG

The persona is therefore a cold-start prior, not an identity:


.:

New User
  
   ↓

TravelProfile
  
   ↓

Same feature vector
  
   ↓

Nearest Persona

   ↓

Collaborative-filter candidates

   ↓

Personalized ranking

In [ ]:
#Score= Confidence × StrengthWeight × SourceWeight

STRENGTH_WEIGHT = {
    "low": 0.3,
    "moderate": 0.65,
    "high": 1.0
}

SOURCE_WEIGHT = {
    "explicit": 1.0,
    "inferred": 0.7
}

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# Existing generated profiles
JSONL_PATH = Path(
"/content/drive/My Drive/IRS Project/Profiles_Out/travel_profiles_QwenLLM.jsonl"

)

print("File exists:", JSONL_PATH.exists())
print("Path:", JSONL_PATH)

profiles = []

with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if line:
            profiles.append(json.loads(line))

print("Profiles loaded:", len(profiles))

File exists: True
Path: /content/drive/My Drive/IRS Project/Profiles_Out/travel_profiles.jsonl
Profiles loaded: 216


In [ ]:
def preference_score(item):

    strength = STRENGTH_WEIGHT[
        item["preference_strength"]
    ]

    source = SOURCE_WEIGHT[
        item["source_type"]
    ]

    return (
        item["confidence"]
        * strength
        * source
    )


def profile_to_features(profile):

    features = {}

    sections = {
        "activities": "activity",
        "interests": "interest",
        "place_preferences": "place",
        "travel_styles": "style"
    }

    for section, prefix in sections.items():

        for item in profile.get(section, []):

            feature = (
                f"{prefix}:{item['concept']}"
            )

            score = preference_score(item)

            # Protect against duplicate concept entries
            features[feature] = max(
                features.get(feature, 0),
                score
            )

    return features

In [ ]:
#create Matrix

feature_rows = []

for profile in profiles:

    row = {
        "profile_id": profile["profile_id"]
    }

    row.update(
        profile_to_features(profile)
    )

    feature_rows.append(row)


feature_df = (
    pd.DataFrame(feature_rows)
    .fillna(0)
    .set_index("profile_id")
)

print(feature_df.shape)

display(feature_df.head())

(216, 57)


,activity:restaurant_dining,interest:food,interest:literature,interest:music,interest:culture,interest:history,style:cultural,style:social,style:nightlife,activity:skiing,...,style:slow_travel,interest:fashion,place:islands,activity:wildlife_viewing,activity:festivals,place:countryside,activity:food_tours,place:forests,place:desert,activity:historical_sites
profile_id,,,,,,,,,,,,,,,,,,,,,
P000001,0.950,0.9800,0.950,0.520,0.95,0.273,0.95000,0.5850,0.27,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
P000002,0.900,0.9800,0.000,0.455,0.00,0.000,0.00000,0.5525,0.00,0.95,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
P000003,0.950,0.9500,0.000,0.990,0.95,0.000,0.96000,0.9000,0.90,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
P000004,0.950,0.6175,0.990,0.950,0.90,0.000,0.59500,0.0000,0.00,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
P000005,0.585,0.5525,0.585,0.950,0.00,0.000,0.34125,0.9500,0.00,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


K-Means now primarily to discover candidate persona archetypes, then use cosine similarity as the actual membership mechanism:

User→{(Persona1 ,0.81),(Persona4,0.69),(Persona
12,0.42)}

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
import pandas as pd

X = normalize(
    feature_df[feature_columns].values,
    norm="l2"
)

for k in range(5, 11):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=30
    )

    labels = model.fit_predict(X)

    print("\n" + "=" * 70)
    print(f"K = {k}")
    print("=" * 70)

    # Cluster sizes
    sizes = pd.Series(labels).value_counts(
        normalize=True
    ).sort_index()

    print("\nCluster proportions:")
    print(sizes.round(3))

    # Centroids
    centroids = pd.DataFrame(
        model.cluster_centers_,
        columns=feature_columns
    )

    print("\nDominant features:")

    for persona_id in range(k):

        top = (
            centroids.iloc[persona_id]
            .sort_values(ascending=False)
            .head(6)
        )

        print(
            f"\nPersona {persona_id}: "
            + ", ".join(top.index)
        )


K = 5

Cluster proportions:
0    0.241
1    0.171
2    0.176
3    0.264
4    0.148
Name: proportion, dtype: float64

Dominant features:

Persona 0: interest:food, interest:music, activity:restaurant_dining, interest:literature, style:social, interest:culture

Persona 1: style:social, interest:food, style:active, activity:restaurant_dining, place:urban, style:adventure

Persona 2: style:social, interest:music, style:nightlife, interest:food, activity:nightlife, place:urban

Persona 3: interest:nature, place:natural_areas, interest:food, style:nature, interest:music, style:active

Persona 4: interest:music, interest:literature, interest:technology, interest:culture, style:cultural, activity:live_music

K = 6

Cluster proportions:
0    0.102
1    0.194
2    0.144
3    0.194
4    0.176
5    0.190
Name: proportion, dtype: float64

Dominant features:

Persona 0: interest:music, interest:literature, interest:technology, interest:culture, interest:history, interest:sports

Persona 1: interest

^explaination of findings above

food

music

social

literature


are common intrests and don't necessarily define a persona.

However, with more features we see music, art, and night life reappearing

.:

the golden question: At what K do we have enough personas to capture meaningful differences without creating redundant personas?

at K=10 we see

social + nightlife + food + music

social + nightlife + music + food

.:

therefore K=6-8 looks like a good area to fall into

Comaring Personas:

calculate cosine to find the similarity of cluster centroids

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import numpy as np
import pandas as pd

X = normalize(
    feature_df[feature_columns].values,
    norm="l2"
)

comparison = []

for k in range(5, 11):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=30
    )

    labels = model.fit_predict(X)

    centroids = model.cluster_centers_

    # similarity between personas
    sim = cosine_similarity(centroids)

    # Ignore self-similarity
    np.fill_diagonal(sim, np.nan)

    comparison.append({
        "k": k,
        "average_persona_similarity": np.nanmean(sim),
        "highest_persona_similarity": np.nanmax(sim),
        "smallest_cluster": pd.Series(labels)
            .value_counts(normalize=True)
            .min()
    })

persona_comparison = pd.DataFrame(comparison)

display(
    persona_comparison.round(3)
)

,k,average_persona_similarity,highest_persona_similarity,smallest_cluster
0,5,0.675,0.807,0.148
1,6,0.654,0.799,0.102
2,7,0.636,0.773,0.083
3,8,0.620,0.836,0.060
4,9,0.613,0.850,0.051
5,10,0.615,0.846,0.056


^ at K=6-7 it shows strong balance where avg similarity is 0.6 and highest similarity (two most similar personas) are less.

.:
lock k=7

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
import pandas as pd

# Keep the original feature columns
feature_columns = [
    c for c in feature_df.columns
    if c != "persona_id"
]

# Build and normalize the matrix
X = feature_df[feature_columns].values
X_normalized = normalize(X, norm="l2")

# FINAL MODEL
N_PERSONAS = 7

persona_model = KMeans(
    n_clusters=N_PERSONAS,
    random_state=42,
    n_init=30
)

# Train
persona_labels = persona_model.fit_predict(X_normalized)

# Assign each historical user to a persona
feature_df["persona_id"] = persona_labels

print("Final model trained with", N_PERSONAS, "personas")

print("\nUsers per persona:")
print(
    feature_df["persona_id"]
    .value_counts()
    .sort_index()
)

Final model trained with 7 personas

Users per persona:
persona_id
0    40
1    31
2    24
3    18
4    32
5    51
6    20
Name: count, dtype: int64


In [ ]:
#save 7 personas centroids

centroids = pd.DataFrame(
    persona_model.cluster_centers_,
    columns=feature_columns
)

centroids.index.name = "persona_id"

display(centroids)

,activity:restaurant_dining,interest:food,interest:literature,interest:music,interest:culture,interest:history,style:cultural,style:social,style:nightlife,activity:skiing,...,style:slow_travel,interest:fashion,place:islands,activity:wildlife_viewing,activity:festivals,place:countryside,activity:food_tours,place:forests,place:desert,activity:historical_sites
persona_id,,,,,,,,,,,,,,,,,,,,,
0,2.726152e-01,0.468774,0.212431,0.273281,0.113779,0.012518,0.069098,0.221207,0.036714,1.029752e-02,...,1.734723e-18,0.021194,2.168404e-18,8.673617e-19,9.809616e-03,0.009340,-3.469447e-18,0.000000e+00,-3.469447e-18,3.551490e-03
1,1.305022e-01,0.174091,0.257961,0.328919,0.210458,0.038400,0.234854,0.156855,0.041188,-6.938894e-18,...,7.826904e-03,0.003099,6.622299e-03,7.420917e-03,1.364592e-02,0.025864,8.028033e-03,3.325640e-02,4.571161e-03,4.671272e-03
2,2.193334e-01,0.254602,0.114854,0.276736,0.054896,0.029771,0.028663,0.239108,0.067271,2.988524e-03,...,1.690306e-03,0.016199,5.955477e-03,8.673617e-19,-1.387779e-17,0.003988,1.244842e-02,1.734723e-18,-1.734723e-18,-1.734723e-18
3,4.367809e-02,0.135501,0.038633,0.103298,0.074996,0.018421,0.059709,0.290904,0.000000,1.040475e-02,...,-8.673617e-19,0.009092,1.492022e-02,8.673617e-19,-6.938894e-18,0.000000,-8.673617e-19,1.734723e-18,-1.734723e-18,-1.301043e-18
4,1.484613e-01,0.260023,0.169146,0.276370,0.125268,0.020579,0.128107,0.330493,0.309670,1.346551e-02,...,6.316215e-03,0.016527,2.168404e-18,1.725629e-02,6.435581e-02,0.001225,1.207611e-02,0.000000e+00,-2.602085e-18,-2.168404e-18
5,1.733375e-01,0.253684,0.116567,0.191094,0.064764,0.007840,0.046924,0.143237,0.047467,5.369798e-02,...,9.457654e-03,0.002653,2.602085e-18,9.098696e-03,1.534258e-02,0.010008,-3.469447e-18,4.353649e-02,1.765934e-02,-2.602085e-18
6,-8.326673e-17,0.006742,0.271815,0.322827,0.076713,0.055210,0.048846,0.029055,0.011695,-6.938894e-18,...,-1.734723e-18,0.028302,1.301043e-18,8.673617e-19,2.606266e-02,0.000000,-1.734723e-18,0.000000e+00,-2.602085e-18,2.211523e-02


In [ ]:
#clean up negatives


centroid_values = np.clip(
    persona_model.cluster_centers_,
    0,
    None
)

centroids = pd.DataFrame(
    centroid_values,
    columns=feature_columns
)

centroids.index.name = "persona_id"

display(centroids.round(4))

,activity:restaurant_dining,interest:food,interest:literature,interest:music,interest:culture,interest:history,style:cultural,style:social,style:nightlife,activity:skiing,...,style:slow_travel,interest:fashion,place:islands,activity:wildlife_viewing,activity:festivals,place:countryside,activity:food_tours,place:forests,place:desert,activity:historical_sites
persona_id,,,,,,,,,,,,,,,,,,,,,
0,0.2726,0.4688,0.2124,0.2733,0.1138,0.0125,0.0691,0.2212,0.0367,0.0103,...,0.0000,0.0212,0.0000,0.0000,0.0098,0.0093,0.0000,0.0000,0.0000,0.0036
1,0.1305,0.1741,0.2580,0.3289,0.2105,0.0384,0.2349,0.1569,0.0412,0.0000,...,0.0078,0.0031,0.0066,0.0074,0.0136,0.0259,0.0080,0.0333,0.0046,0.0047
2,0.2193,0.2546,0.1149,0.2767,0.0549,0.0298,0.0287,0.2391,0.0673,0.0030,...,0.0017,0.0162,0.0060,0.0000,0.0000,0.0040,0.0124,0.0000,0.0000,0.0000
3,0.0437,0.1355,0.0386,0.1033,0.0750,0.0184,0.0597,0.2909,0.0000,0.0104,...,0.0000,0.0091,0.0149,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,0.1485,0.2600,0.1691,0.2764,0.1253,0.0206,0.1281,0.3305,0.3097,0.0135,...,0.0063,0.0165,0.0000,0.0173,0.0644,0.0012,0.0121,0.0000,0.0000,0.0000
5,0.1733,0.2537,0.1166,0.1911,0.0648,0.0078,0.0469,0.1432,0.0475,0.0537,...,0.0095,0.0027,0.0000,0.0091,0.0153,0.0100,0.0000,0.0435,0.0177,0.0000
6,0.0000,0.0067,0.2718,0.3228,0.0767,0.0552,0.0488,0.0291,0.0117,0.0000,...,0.0000,0.0283,0.0000,0.0000,0.0261,0.0000,0.0000,0.0000,0.0000,0.0221


| Persona | Share |
| ------- | ----: |
| 0       | 18.5% |
| 1       | 14.4% |
| 2       | 11.1% |
| 3       |  8.3% |
| 4       | 14.8% |
| 5       | 23.6% |
| 6       |  9.3% |


In [ ]:
from pathlib import Path
import joblib

# Define output directory
OUTPUT_DIR = Path(
    "/content/drive/My Drive/IRS Project/Profiles_Out"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Save final K=7 model
MODEL_PATH = OUTPUT_DIR / "travel_persona_kmeans_k7.joblib"

joblib.dump(
    {
        "model": persona_model,
        "feature_columns": feature_columns
    },
    MODEL_PATH
)

print("Saved:", MODEL_PATH)



Saved: /content/drive/My Drive/IRS Project/Profiles_Out/travel_persona_kmeans_k7.joblib


In [ ]:
CENTROIDS_PATH = OUTPUT_DIR / "travel_persona_centroids_k7.csv"

centroids.to_csv(CENTROIDS_PATH)

print("Saved:", CENTROIDS_PATH)

Saved: /content/drive/My Drive/IRS Project/Profiles_Out/travel_persona_centroids_k7.csv
